In [ ]:
import math
import numpy as np
import json
import pickle
import os.path
from os import path
import shutil
import subprocess
import pymongo
import uuid
import pandas as pd
from pulp import *
import xlrd
from flask import Flask, request, jsonify
from flask_cors import CORS
import mysql.connector
import requests
import time
import secrets
import string
from datetime import datetime
from pulp.constants import LpStatus, LpStatusInfeasible, LpStatusUnbounded, LpStatusNotSolved, LpStatusUndefined



from io import BytesIO
from cryptography.fernet import Fernet		
import os
from datetime import datetime

input = pd.ExcelFile('Backend//Data_11.xlsx')
node1 = pd.read_excel(input,sheet_name="A.1 Warehouse")
node2 = pd.read_excel(input,sheet_name="A.2 FPS")
node3 = pd.read_excel(input,sheet_name="A.2 Warehouse")
node4 = pd.read_excel(input,sheet_name="A.3 Storage")
node5 = pd.read_excel(input,sheet_name="A.3 RH")
df_map = pd.read_excel(input,sheet_name="RHMap")
# node1 = pd.read_excel(input,sheet_name="A.1 Warehouse")
# node2 = pd.read_excel(input,sheet_name="A.2 FPS")
# import math

# import math
# import pandas as pd

# # Load Excel File
# input = pd.ExcelFile('Backend/Data_11.xlsx')

# # Load sheets into DataFrames with clear names
# FCI = pd.read_excel(input, sheet_name="A.1 Warehouse")   # FCI warehouses
# FPS = pd.read_excel(input, sheet_name="A.2 FPS")         # FPS outlets
# SP  = pd.read_excel(input, sheet_name="A.2 Warehouse")   # State procurement warehouses
# WH  = pd.read_excel(input, sheet_name="A.3 Storage")     # Storage warehouses

import math

R = 6371  # Earth radius in km

# 1. FCI (node1) → FPS (node2)
dist = {}
for i in node1.index:
    for j in node2.index:
        phi_1 = math.radians(node1["WH_Lat"][i])
        phi_2 = math.radians(node2["FPS_Lat"][j])
        delta_phi = math.radians(node2["FPS_Lat"][j] - node1["WH_Lat"][i])
        delta_lambda = math.radians(node2["FPS_Long"][j] - node1["WH_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist[(i, j)] = R * c

# 2. FCI (node1) → SP (node4)
dist1 = {}
for i in node1.index:
    for j in node4.index:
        phi_1 = math.radians(node1["WH_Lat"][i])
        phi_2 = math.radians(node4["WH_Lat"][j])
        delta_phi = math.radians(node4["WH_Lat"][j] - node1["WH_Lat"][i])
        delta_lambda = math.radians(node4["WH_Long"][j] - node1["WH_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist1[(i, j)] = R * c

# 3. FPS (node2) → SW (node3)
dist2 = {}
for i in node2.index:
    for j in node3.index:
        phi_1 = math.radians(node2["FPS_Lat"][i])
        phi_2 = math.radians(node3["SW_Lat"][j])
        delta_phi = math.radians(node3["SW_Lat"][j] - node2["FPS_Lat"][i])
        delta_lambda = math.radians(node3["SW_Long"][j] - node2["FPS_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist2[(i, j)] = R * c

# 4. SP (node4) → FPS (node2)
dist3 = {}
for i in node4.index:
    for j in node2.index:
        phi_1 = math.radians(node4["WH_Lat"][i])
        phi_2 = math.radians(node2["FPS_Lat"][j])
        delta_phi = math.radians(node2["FPS_Lat"][j] - node4["WH_Lat"][i])
        delta_lambda = math.radians(node2["FPS_Long"][j] - node4["WH_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist3[(i, j)] = R * c

# 5. FCI (node1) → RH (node5)
dist4 = {}
for i in node1.index:
    for j in node5.index:
        phi_1 = math.radians(node1["WH_Lat"][i])
        phi_2 = math.radians(node5["WH_Lat"][j])
        delta_phi = math.radians(node5["WH_Lat"][j] - node1["WH_Lat"][i])
        delta_lambda = math.radians(node5["WH_Long"][j] - node1["WH_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist4[(i, j)] = R * c


 # 6. RH (node5) → SP (node4) 
dist5 = {}
for i in node5.index:
    for j in node4.index:
        phi_1 = math.radians(node5["WH_Lat"][i])
        phi_2 = math.radians(node4["WH_Lat"][j])
        delta_phi = math.radians(node4["WH_Lat"][j] - node5["WH_Lat"][i])
        delta_lambda = math.radians(node4["WH_Long"][j] - node5["WH_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist5[(i, j)] = R * c


 # 7. RH (node5) → FPS (node2)
dist6 = {}
for i in node5.index:
    for j in node2.index:
        phi_1 = math.radians(node5["WH_Lat"][i])
        phi_2 = math.radians(node2["FPS_Lat"][j])
        delta_phi = math.radians(node2["FPS_Lat"][j] - node5["WH_Lat"][i])
        delta_lambda = math.radians(node2["FPS_Long"][j] - node5["WH_Long"][i])

        a = math.sin(delta_phi / 2.0)**2 + math.cos(phi_1) * math.cos(phi_2) * math.sin(delta_lambda / 2.0)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        dist6[(i, j)] = R * c

     

  


In [2]:
print(list(dist1.keys())[:10])
print(type(dist1))

[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]
<class 'dict'>


In [11]:
USN = pd.ExcelFile('Backend//Data_11.xlsx')
FCI = pd.read_excel(USN, sheet_name='A.1 Warehouse', index_col=None)
FPS = pd.read_excel(USN, sheet_name='A.2 FPS', index_col=None)
SP = pd.read_excel(USN, sheet_name='A.3 Storage', index_col=None)
WH = pd.read_excel(USN, sheet_name='A.2 Warehouse', index_col=None)
RH = pd.read_excel(USN, sheet_name='A.3 RH', index_col=None)

# remove all the spaces in the district columns of all dataframes
FCI['WH_District'] = FCI['WH_District'].str.lower().str.replace(r'\s+', '', regex=True)
FPS['FPS_District'] = FPS['FPS_District'].str.lower().str.replace(r'\s+', '', regex=True)
WH['SW_District'] = WH['SW_District'].str.lower().str.replace(r'\s+', '', regex=True)
SP['WH_District'] = SP['WH_District'].str.lower().str.replace(r'\s+', '', regex=True)
RH['WH_District'] = RH['WH_District'].str.lower().str.replace(r'\s+', '', regex=True)



In [12]:
from pulp import LpVariable
model = LpProblem('Supply-Demand-Problem', LpMinimize)

surplus_districts = set(['thanjavur', 'thiruvarur'])
deficit_districts = set(['thiruchirappalli', 'virudhunagar'])

In [5]:
# -----------------------
# 1. FCI → FPS (Allocation1)
# -----------------------
Allocation1 = {}
for i in range(len(FCI['WH_ID'])):
    for j in range(len(FPS['FPS_ID'])):
        from_district = FCI['WH_District'][i]
        to_district = FPS['FPS_District'][j]
        var_name1 = f"X_{FCI['WH_ID'][i]}_{from_district}_{FPS['FPS_ID'][j]}_{to_district}_Wheat"
        Allocation1[(i, j)] = LpVariable(var_name1, lowBound=0, cat='Continuous')

In [14]:
Allocation1

{(0, 0): X_P17010811_Ariyalur_H17010552_Ariyalur_Wheat,
 (0, 1): X_P17010811_Ariyalur_H17010352_Ariyalur_Wheat,
 (0, 2): X_P17010811_Ariyalur_H17011191_Ariyalur_Wheat,
 (0, 3): X_P17010811_Ariyalur_H35070341_Chengalpattu_Wheat,
 (0, 4): X_P17010811_Ariyalur_H35111521_Chengalpattu_Wheat,
 (0, 5): X_P17010811_Ariyalur_H35062521_Chengalpattu_Wheat,
 (0, 6): X_P17010811_Ariyalur_H35062491_Chengalpattu_Wheat,
 (0, 7): X_P17010811_Ariyalur_H35071501_Chengalpattu_Wheat,
 (0, 8): X_P17010811_Ariyalur_H35111393_Chengalpattu_Wheat,
 (0, 9): X_P17010811_Ariyalur_H35062391_Chengalpattu_Wheat,
 (0, 10): X_P17010811_Ariyalur_H35111391_Chengalpattu_Wheat,
 (0, 11): X_P17010811_Ariyalur_H35111392_Chengalpattu_Wheat,
 (0, 12): X_P17010811_Ariyalur_H3511139102_Chengalpattu_Wheat,
 (0, 13): X_P17010811_Ariyalur_H35071502_Chengalpattu_Wheat,
 (0, 14): X_P17010811_Ariyalur_H35043291_Chengalpattu_Wheat,
 (0, 15): X_P17010811_Ariyalur_H560010051_ChennaiN_Wheat,
 (0, 16): X_P17010811_Ariyalur_H560010052_Chenn

In [ ]:
print(len(FCI), len(FPS), len(Allocation1))

2508 621 1557468


In [ ]:
# -----------------------
# 2. FCI → SP (Allocation2)
# -----------------------
Allocation2 = {}
for i in range(len(FCI['WH_ID'])):
    for j in range(len(SP['WH_ID'])):
        from_district = FCI['WH_District'][i]
        to_district = SP['WH_District'][j]

        # if (from_district in surplus_districts and to_district in surplus_districts) or \
        #    (from_district in deficit_districts and to_district in deficit_districts):

        var_name2 = f"Y_{FCI['WH_ID'][i]}_{from_district}_{SP['WH_ID'][j]}_{to_district}_Wheat"
        Allocation2[(i, j)] = LpVariable(var_name2, lowBound=0, cat='Continuous')

# -----------------------
# 3. SP → FPS (Allocation3)
# -----------------------
Allocation3 = {}
for i in range(len(SP['WH_ID'])):
    for j in range(len(FPS['FPS_ID'])):
        from_district = SP['WH_District'][i]
        to_district = FPS['FPS_District'][j]

        # if (from_district in surplus_districts and to_district in surplus_districts) or \
        #    (from_district in deficit_districts and to_district in deficit_districts):

        var_name3 = f"Z_{SP['WH_ID'][i]}_{from_district}_{FPS['FPS_ID'][j]}_{to_district}_Wheat"
        Allocation3[(i, j)] = LpVariable(var_name3, lowBound=0, cat='Continuous')

# -----------------------
# 4. FPS → SW (Allocation4)
# -----------------------
Allocation4 = {}
for i in range(len(FPS['FPS_ID'])):
    for j in range(len(WH['SW_ID'])):
        from_district = FPS['FPS_District'][i]
        to_district = WH['SW_District'][j]

        # if (from_district in surplus_districts and to_district in surplus_districts) or \
        #    (from_district in deficit_districts and to_district in deficit_districts):

        var_name4 = f"W_{FPS['FPS_ID'][i]}_{from_district}_{WH['SW_ID'][j]}_{to_district}_Wheat"
        Allocation4[(i, j)] = LpVariable(var_name4, lowBound=0, cat='Continuous')


# -----------------------
# 5. FCI → RH (Allocation5)
# -----------------------
Allocation5 = {}
for i in range(len(FCI['WH_ID'])):
    for j in range(len(RH['WH_ID'])):
        from_district = FCI['WH_District'][i]
        to_district = RH['WH_District'][j]

        # if (from_district in surplus_districts and to_district in surplus_districts) or \
        #    (from_district in deficit_districts and to_district in deficit_districts):

        var_name5 = f"A_{FCI['WH_ID'][i]}_{from_district}_{RH['WH_ID'][j]}_{to_district}_Wheat"
        Allocation5[(i, j)] = LpVariable(var_name5, lowBound=0, cat='Continuous')


# -----------------------
# 6. RH (node5) → SP (node3)
# -----------------------
Allocation6 = {}
for i in range(len(RH['WH_ID'])):
    for j in range(len(SP['WH_ID'])):
        from_district = RH['WH_District'][i]
        to_district = SP['WH_District'][j]

        # if (from_district in surplus_districts and to_district in surplus_districts) or \
        #    (from_district in deficit_districts and to_district in deficit_districts):

        var_name6 = f"B_{RH['WH_ID'][i]}_{from_district}_{SP['WH_ID'][j]}_{to_district}_Wheat"
        Allocation6[(i, j)] = LpVariable(var_name6, lowBound=0, cat='Continuous')


# -----------------------
# 7. RH (node5) → FPS (node2)
# -----------------------
Allocation7 = {}
for i in range(len(RH['WH_ID'])):
    for j in range(len(FPS['FPS_ID'])):
        from_district = RH['WH_District'][i]
        to_district = FPS['FPS_District'][j]

        # if (from_district in surplus_districts and to_district in surplus_districts) or \
        #    (from_district in deficit_districts and to_district in deficit_districts):

        var_name7 = f"C_{RH['WH_ID'][i]}_{from_district}_{FPS['FPS_ID'][j]}_{to_district}_Wheat"
        Allocation7[(i, j)] = LpVariable(var_name7, lowBound=0, cat='Continuous')



In [8]:
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, CPLEX_CMD, LpStatus

# Create the model
model = LpProblem("Wheat_Distribution_Model", LpMinimize)

objective = lpSum(Allocation1[(i, j)] * dist[(i, j)] for (i, j) in Allocation1) + \
            lpSum(Allocation2[(i, j)] * dist1[(i, j)] for (i, j) in Allocation2) + \
            lpSum(Allocation3[(i, j)] * dist3[(i, j)] for (i, j) in Allocation3) + \
            lpSum(Allocation4[(i, j)] * dist2[(i, j)] for (i, j) in Allocation4) +\
            lpSum(Allocation5[(i, j)] * dist4[(i, j)] for (i, j) in Allocation5) +\
            lpSum(Allocation6[(i, j)] * dist5[(i, j)] for (i, j) in Allocation6) +\
            lpSum(Allocation7[(i, j)] * dist6[(i, j)] for (i, j) in Allocation7) 
          
model += objective, "Total_Transport_Distance"
# a = 0.225
# a = 0.0125
# coefficient

# --- Rename columns as needed ---
FCI.rename(columns={'Paddy_Procurement': 'Storage_Capacity'}, inplace=True)
FPS.rename(columns={'Milling_Process': 'Allocation_Wheat', 'Mill_Capacity': 'Capacity'}, inplace=True)

FPS['Demand'] = FPS['Allocation_Wheat']
rh_mapping_pairs = list(zip(df_map['From_RH'], df_map['To_RH']))
Allocation8 = LpVariable.dicts("RH_RH", rh_mapping_pairs, lowBound=0)
valid_from_RH = set(i for i, j in rh_mapping_pairs)

for key in Allocation5:
    i, j = key

    if j not in valid_from_RH:
        model += Allocation5[key] == 0
# # Surplus and deficit districts
# # surplus_districts = set(['Thanjavur', 'Thiruvarur'])
# # deficit_districts = set(['Thiruchirappalli', 'Virudhunagar'])

# # All allocations
# # Define your surplus districts
# # List of all allocation variable sets
# # alloc_sets = [Allocation1, Allocation2, Allocation3, Allocation4]

# # for alloc in alloc_sets:
# #     keys_to_remove = []
# #     for key, var in alloc.items():
# #         var_name = str(var)
# #         parts = var_name.split("_")
# #         wh_district = parts[2]
# #         fps_district = parts[4]

# #         if not (wh_district in surplus_districts and fps_district in surplus_districts):
# #             model += var == 0
# #             keys_to_remove.append(key)
    
# #     # Actually remove them
# #     for k in keys_to_remove:
# #         del alloc[k]
# # print(Allocation1)
# # --- Constraint: Surplus flow rules ---
#     # FPS demand constraints
# surplus_districts = ["Ariyalur", "Chengalpattu", "Cuddalore", "Kancheepuram",
#                      "Mayiladuthurai", "Nagapattinam", "Pudukkottai", "Ranipet",
#                      "Thanjavur", "Thiruvallore", "Thiruvanamalai", "Thiruvarur",
#                      "Thiruchirappalli"]

# # --- Unified Surplus + Deficit Constraints ---

# # Define districts
# deficit_districts = ["ChennaiN", "ChennaiS", "Coimbatore", "Dharmapuri",
#                       "Dindigul", "Erode", "Kallakurichi", "Kanyakumari", "Karur", "Krishnagiri",
#                       "Madurai", "Namakkal", "Perambalur", "Salem", "Sivagangai", "Tenkasi", "Theni",
#                       "Thirunelveli", "Tiruppur", "Tirupathur", "Vellore", "Villupuram", "Virudhunagar",
#                       "Ramanathapuram", "Thoothukudi", "Nilgiris"]
# a = 0.225  # coefficient

# # --- FPS Allocation & Milling Constraints ---
# for i in range(len(FPS['FPS_ID'])):
#     district = FPS.loc[i, 'FPS_District']
    
#     if district in surplus_districts:
#         # Surplus FPS constraints
#         model += lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) == a * FPS.loc[i, 'Demand']
#         total_allocated = lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) + \
#                           lpSum(Allocation3[key] for key in Allocation3 if key[1] == i)
#         model += total_allocated <= FPS.loc[i, 'Capacity']

#     elif district in deficit_districts:
#         # Deficit FPS constraints (can adjust coefficient if needed)
#         model += lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) <= a * FPS.loc[i, 'Demand']
#         total_allocated = lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) + \
#                           lpSum(Allocation3[key] for key in Allocation3 if key[1] == i)
#         model += total_allocated <= FPS.loc[i, 'Capacity']

# # --- SP Capacity Constraints ---
# for i in range(len(SP['WH_ID'])):
#     district = SP.loc[i, 'WH_District']
    
#     if district in surplus_districts:
#         model += lpSum(Allocation2[key] for key in Allocation2 if key[1] == i) <= SP.loc[i, 'Capacity']
    
#     elif district in deficit_districts:
#         model += lpSum(Allocation2[key] for key in Allocation2 if key[1] == i) <= SP.loc[i, 'Capacity']

# # --- FCI Storage Constraints ---
# for i in range(len(FCI['WH_ID'])):
#     district = FCI.loc[i, 'WH_District']
    
#     if district in surplus_districts or district in deficit_districts:
#         model += (
#             lpSum(Allocation1[key] for key in Allocation1 if key[0] == i) +
#             lpSum(Allocation2[key] for key in Allocation2 if key[0] == i)
#         ) <= FCI.loc[i, 'Storage_Capacity']

# # --- FPS Allocation to Warehouse Relations ---
# for j in range(len(FPS['FPS_ID'])):
#     district = FPS.loc[j, 'FPS_District']
    
#     if district in surplus_districts or district in deficit_districts:
#         model += (
#             lpSum(Allocation4[key] for key in Allocation4 if key[0] == j)
#         ) == 0.67 * (
#             lpSum(Allocation1[key] for key in Allocation1 if key[1] == j) +
#             lpSum(Allocation3[key] for key in Allocation3 if key[1] == j)
#         )

# # --- SP to FPS Flow Limit (Generation only for deficit districts) ---
# for k in range(len(SP['WH_ID'])):
#     district = SP.loc[k, 'WH_District']
    
#     if district in surplus_districts:
#         # Surplus: allocation3 = allocation2 (no generation)
#         model += (
#             lpSum(Allocation3[key] for key in Allocation3 if key[0] == k)
#         ) <= lpSum(Allocation2[key] for key in Allocation2 if key[1] == k)
    
#     elif district in deficit_districts:
#         # Deficit: allocation3 = allocation2 + generation
#         model += (
#             lpSum(Allocation3[key] for key in Allocation3 if key[0] == k)
#         ) <= (
#             lpSum(Allocation2[key] for key in Allocation2 if key[1] == k) +
#             Generation[k]
#         )

# # --- Warehouse Demand Fulfillment ---
# for m in range(len(WH['SW_ID'])):
#     district = WH.loc[m, 'SW_District']
    
#     if district in surplus_districts or district in deficit_districts:
#         model += (
#             lpSum(Allocation4[key] for key in Allocation4 if key[1] == m)
#         )  == WH['Annual_Demand'][m]

# --- Define Sets ---
surplus_districts = [
    "Ariyalur", "Chengalpattu", "Cuddalore", "Kancheepuram",
    "Mayiladuthurai", "Nagapattinam", "Pudukkottai", "Ranipet",
    "Thanjavur", "Thiruvallur", "Thiruvanamalai", "Thiruvarur",
    "Thiruchirappalli"
]

deficit_districts = [
    "ChennaiN", "ChennaiS", "Coimbatore", "Dharmapuri",
    "Dindigul", "Erode", "Kallakurichi", "Kanyakumari", "Karur", "Krishnagiri",
    "Madurai", "Namakkal", "Perambalur", "Salem", "Sivagangai", "Tenkasi", "Theni",
    "Thirunelveli", "Tiruppur", "Tirupathur", "Vellore", "Villupuram", "Virudhunagar",
    "Ramanathapuram", "Thoothukudi", "Nilgiris"
]

a = 0.225  # conversion coefficient
# a = 0.0125

# ===============================================================
# 1️⃣ FPS Allocation & Milling Capacity Constraints
# ===============================================================
for i in range(len(FPS['FPS_ID'])):
    fps_district = FPS.loc[i, 'FPS_District']

    # --- Surplus FPS ---
    if fps_district in surplus_districts:
        model += lpSum(
            Allocation1[key] for key in Allocation1
            if key[1] == i and FCI.loc[key[0], 'WH_District'] in surplus_districts
        ) == a * FPS.loc[i, 'Demand']

        total_allocated = lpSum(
            Allocation1[key] for key in Allocation1
            if key[1] == i and FCI.loc[key[0], 'WH_District'] in surplus_districts
        ) + lpSum(
            Allocation3[key] for key in Allocation3
            if key[1] == i and SP.loc[key[0], 'WH_District'] in surplus_districts
        )

        model += total_allocated <= FPS.loc[i, 'Capacity']

    # --- Deficit FPS ---
    elif fps_district in deficit_districts:
        model += lpSum(
            Allocation1[key] for key in Allocation1
            if key[1] == i and FCI.loc[key[0], 'WH_District'] in deficit_districts
        ) <= a * FPS.loc[i, 'Demand']

        total_allocated = lpSum(
            Allocation1[key] for key in Allocation1
            if key[1] == i and FCI.loc[key[0], 'WH_District'] in deficit_districts
        ) + lpSum(
            Allocation3[key] for key in Allocation3
            if key[1] == i and SP.loc[key[0], 'WH_District'] in deficit_districts
        ) + lpSum(
            Allocation7[key] for key in Allocation7
            if key[1] == i and RH.loc[key[0], 'WH_District'] in deficit_districts
        )
        model += total_allocated <= FPS.loc[i, 'Capacity']

        
# ===============================================================
# 2️⃣ SP (Storage Point) Capacity Constraints
# ===============================================================
for j in range(len(SP['WH_ID'])):
    sp_district = SP.loc[j, 'WH_District']

    if sp_district in surplus_districts:
        # Surplus storage points — only receive from surplus PCs
        model += lpSum(
            Allocation2[key] for key in Allocation2
            if key[1] == j and FCI.loc[key[0], 'WH_District'] in surplus_districts
        ) <= SP.loc[j, 'Capacity']

    elif sp_district in deficit_districts:
        # Deficit storage points — can receive from:
        # (1) deficit PCs (same-region)
        # (2) surplus PCs within 170 km
        # (3) deficit Railheads
        model += lpSum(
            Allocation2[key] for key in Allocation2
            if key[1] == j and (
                FCI.loc[key[0], 'WH_District'] in deficit_districts or
                (FCI.loc[key[0], 'WH_District'] in surplus_districts and dist1.get((key[0], j)) <= 150)
            )
        ) + lpSum(
            Allocation6[key] for key in Allocation6
            if key[1] == j and RH.loc[key[0], 'WH_District'] in deficit_districts
        ) <= SP.loc[j, 'Capacity']




# ===============================================================
# 3️⃣ FCI (Procurement Center) Storage Constraints
# ===============================================================
# ===============================================================
# 3️⃣ FCI (Procurement Center) Storage Constraints – Surplus→Surplus and Deficit→Deficit only
# ===============================================================
# for i in range(len(FCI['WH_ID'])):
#     pc_district = FCI.loc[i, 'WH_District']

#     # --- Surplus PCs: can send to surplus mills, surplus storages, and nearby deficit storages (≤170 km) ---
#     if pc_district in surplus_districts:
#         model += (
#             lpSum(
#                 Allocation1[key] for key in Allocation1
        #         if key[0] == i and FPS.loc[key[1], 'FPS_District'] in surplus_districts
        #     ) +
        #     lpSum(
        #         Allocation2[key] for key in Allocation2
        #         if key[0] == i and (
        #             SP.loc[key[1], 'WH_District'] in surplus_districts or
        #             (
        #                 SP.loc[key[1], 'WH_District'] in deficit_districts and
        #                 dist1.get((i, key[1])) <= 170
        #             )
        #         )
        #     )
        # ) <= FCI.loc[i, 'Storage_Capacity']


    # --- Surplus PCs: can send to surplus mills, surplus storages, and nearby deficit storages (≤170 km) ---
for i in range(len(FCI['WH_ID'])):
    pc_district = FCI.loc[i, 'WH_District']

    # --- Surplus PCs: can send to surplus mills, surplus storages, nearby deficit storages (≤170 km), and surplus railheads ---
    if pc_district in surplus_districts:
        model += (
            # --- PC → Mill (only to surplus mills) ---
            lpSum(
                Allocation1[key] for key in Allocation1
                if key[0] == i and FPS.loc[key[1], 'FPS_District'] in surplus_districts
            )
            +
            # --- PC → Storage (only to surplus storages) ---
            lpSum(
                Allocation2[key] for key in Allocation2
                if key[0] == i and SP.loc[key[1], 'WH_District'] in surplus_districts
            )
            +
            # --- PC → Storage (to deficit storages within 170 km) ---
            lpSum(
                Allocation2[key] for key in Allocation2
                if key[0] == i and SP.loc[key[1], 'WH_District'] in deficit_districts
                and dist1.get((i, key[1])) <= 150
            )
            +
            # --- PC → Railhead (only to surplus railheads) ---
            lpSum(
                Allocation5[key] for key in Allocation5
                if key[0] == i and RH.loc[key[1], 'WH_District'] in surplus_districts
            )
        ) <= FCI.loc[i, 'Storage_Capacity']

    # --- Deficit PCs: can send only to deficit mills and deficit storages ---
    elif pc_district in deficit_districts:
        model += (
            # --- PC → Mill (only to deficit mills) ---
            lpSum(
                Allocation1[key] for key in Allocation1
                if key[0] == i and FPS.loc[key[1], 'FPS_District'] in deficit_districts
            )
            +
            # --- PC → Storage (only to deficit storages) ---
            lpSum(
                Allocation2[key] for key in Allocation2
                if key[0] == i and SP.loc[key[1], 'WH_District'] in deficit_districts
            )
        ) == FCI.loc[i, 'Storage_Capacity']

           
    # --- Surplus PCs: can send to surplus mills, surplus storages, nearby deficit storages (≤170 km), and surplus rail heads ---

    # --- Deficit PCs: can send only to deficit mills and storages ---
   

# ===============================================================
# 4️⃣ FPS to Warehouse Relation (Mill to WH)
# ===============================================================
# ===============================================================
# 4️⃣ FPS to Warehouse Relation (Mill to WH) – Within-Type Only
# ===============================================================
for j in range(len(FPS['FPS_ID'])):
    fps_district = FPS.loc[j, 'FPS_District']

    # --- Surplus mills: send only to surplus warehouses ---
    if fps_district in surplus_districts:
        model += (
            lpSum(
                Allocation4[key] for key in Allocation4
                if key[0] == j and WH.loc[key[1], 'SW_District'] in surplus_districts
            )
        ) == 0.68 * (
            lpSum(
                Allocation1[key] for key in Allocation1
                if key[1] == j and FCI.loc[key[0], 'WH_District'] in surplus_districts
            ) +
            lpSum(
                Allocation3[key] for key in Allocation3
                if key[1] == j and SP.loc[key[0], 'WH_District'] in surplus_districts
            )
        )

    # --- Deficit mills: send only to deficit warehouses ---
    elif fps_district in deficit_districts:
      model += (
        lpSum(
            Allocation4[key] for key in Allocation4
            if key[0] == j and WH.loc[key[1], 'SW_District'] in deficit_districts
        )
    ) == 0.68 * (
        lpSum(
            Allocation1[key] for key in Allocation1
            if key[1] == j and FCI.loc[key[0], 'WH_District'] in deficit_districts
        ) +
        lpSum(
            Allocation3[key] for key in Allocation3
            if key[1] == j and SP.loc[key[0], 'WH_District'] in deficit_districts
        ) +
        lpSum(
            Allocation7[key] for key in Allocation7
            if key[1] == j and RH.loc[key[0], 'WH_District'] in deficit_districts
        )
    )


# ===============================================================
# 5️⃣ SP to FPS Flow (Generation for Deficit)
# ===============================================================
# ===============================================================
# 5️⃣ SP to Mill / Storage Relation (Storage to Mill) – Within-Type Only
# ===============================================================
for k in range(len(SP['WH_ID'])):
    sp_district = SP.loc[k, 'WH_District']

    # --- Surplus SP: can send only based on surplus allocations ---
    if sp_district in surplus_districts:
        model += (
            lpSum(
                Allocation3[key] for key in Allocation3
                if key[0] == k and FPS.loc[key[1], 'FPS_District'] in surplus_districts
            )
        ) <= lpSum(
            Allocation2[key] for key in Allocation2
            if key[1] == k and FCI.loc[key[0], 'WH_District'] in surplus_districts
        )

    # --- Deficit SP: can send based on allocations from surplus PC + rail heads + generation ---
    elif sp_district in deficit_districts:
      model += (
        lpSum(
            Allocation3[key] for key in Allocation3
            if key[0] == k and FPS.loc[key[1], 'FPS_District'] in deficit_districts
        )
    ) == (
        lpSum(
            Allocation2[key] for key in Allocation2
            if key[1] == k and FCI.loc[key[0], 'WH_District'] in deficit_districts
        ) +
        lpSum(
            Allocation6[key] for key in Allocation6
            if key[1] == k and RH.loc[key[0], 'WH_District'] in deficit_districts
        ) +
        lpSum(
            Allocation2[key] for key in Allocation2
            if key[1] == k and FCI.loc[key[0], 'WH_District'] in surplus_districts and
               dist1.get((key[0], k), 99999) <= 150
        )
    )



# ===============================================================
# 6️⃣ Warehouse Demand Fulfillment
# ===============================================================
# ===============================================================
# 6️⃣ Warehouse Demand Fulfillment – Surplus vs Deficit
# ===============================================================
for m in range(len(WH['SW_ID'])):
    wh_district = WH.loc[m, 'SW_District']

    # --- Surplus warehouses: receive only from surplus mills ---
    if wh_district in surplus_districts:
        model += lpSum(
            Allocation4[key] for key in Allocation4
            if key[1] == m and FPS.loc[key[0], 'FPS_District'] in surplus_districts
        ) == WH.loc[m, 'Annual_Demand']

    # --- Deficit warehouses: receive only from deficit mills ---
    elif wh_district in deficit_districts:
        model += lpSum(
            Allocation4[key] for key in Allocation4
            if key[1] == m and FPS.loc[key[0], 'FPS_District'] in deficit_districts
        ) == WH.loc[m, 'Annual_Demand']



# for r in range(len(RH)):
#     model += (
#         lpSum(Allocation5[key] for key in Allocation5 if key[1] == r)
#         ==
#         lpSum(Allocation6[key] for key in Allocation6 if key[0] == r) +
#         lpSum(Allocation7[key] for key in Allocation7 if key[0] == r)
#     )

# for key in Allocation5:
#     i, j = key
#     if not (
#         FCI.loc[i, 'WH_District'] in surplus_districts and
#         RH.loc[j, 'WH_District'] in surplus_districts
#     ):
#         model += Allocation5[key] == 0

# for key in Allocation6:
#     i, j = key
#     if not (
#         RH.loc[i, 'WH_District'] in deficit_districts and
#         FPS.loc[j, 'FPS_District'] in deficit_districts
#     ):
#         model += Allocation6[key] == 0


# for key in Allocation7:
#     i, j = key
    # if not (
    #     RH.loc[i, 'WH_District'] in deficit_districts and
    #     SP.loc[j, 'WH_District'] in deficit_districts
    # ):
    #     model += Allocation7[key] == 0
# ===============================================================
# Rail Head Flow Balance: Allocation5 (PC → Rail Head) = Allocation6 + Allocation7
# ===============================================================
# ===============================================================
# 🔄 Railhead Balance: Surplus Outflow = Deficit Inflow
# ===============================================================

# ===============================================================
# 🔄 Railhead Balance: Surplus Outflow = Deficit Inflow
# ===============================================================

# Railhead balance: Surplus outflow = Deficit inflow
# --- Total inflow to SURPLUS railheads (from surplus PCs) ---
# total_inflow_surplus_RH = lpSum(
#     Allocation5[key]
#     for key in Allocation5
#     if (
#         key[0] in FCI.index and
#         key[1] in RH.index and
#         FCI.loc[key[0], 'WH_District'] in surplus_districts and
#         RH.loc[key[1], 'WH_District'] in surplus_districts
#     )
# )

# # --- Total outflow from DEFICIT railheads (to deficit Mills + Storages) ---
# total_outflow_deficit_RH = (
#     lpSum(
#         Allocation6[key]
#         for key in Allocation6
#         if (
#             key[0] in RH.index and
#             key[1] in FPS.index and
#             RH.loc[key[0], 'WH_District'] in deficit_districts and
#             FPS.loc[key[1], 'FPS_District'] in deficit_districts
#         )
#     )
#     +
#     lpSum(
#         Allocation7[key]
#         for key in Allocation7
#         if (
#             key[0] in RH.index and
#             key[1] in SP.index and
#             RH.loc[key[0], 'WH_District'] in deficit_districts and
#             SP.loc[key[1], 'WH_District'] in deficit_districts
#         )
#     )
# )

# # --- Balance constraint ---
# model += total_inflow_surplus_RH == total_outflow_deficit_RH, "Railhead_Balance"

# for r in range(len(RH)):
#     rh_district = RH.loc[r, 'WH_District']
    
#     if rh_district in surplus_districts:
#         # Railhead can only receive from surplus PCs (Allocation5)
#         model += (
#             lpSum(Allocation5[key] for key in Allocation5 if key[1] == r)
#         ) <= RH.loc[r, 'Capacity']
        
#     elif rh_district in deficit_districts:
#         # Railhead can only send to deficit mills and storages (Alloc6, Alloc7)
#         model += (
#             lpSum(Allocation6[key] for key in Allocation6 if key[0] == r) +
#             lpSum(Allocation7[key] for key in Allocation7 if key[0] == r)
# --- Surplus RH ---
for i in RH.index:
    if RH.loc[i, 'WH_District'] in surplus_districts:

        model += lpSum(
            Allocation8[(i, j)] for (i2, j) in rh_mapping_pairs if i2 == i
        ) == lpSum(
            Allocation5[key] for key in Allocation5 if key[1] == i
        )

# --- Deficit RH ---
for j in RH.index:
    if RH.loc[j, 'WH_District'] in deficit_districts:

        model += (
            lpSum(Allocation6[key] for key in Allocation6 if key[0] == j) +
            lpSum(Allocation7[key] for key in Allocation7 if key[0] == j)
        ) == lpSum(
            Allocation8[(i, j)] for (i, j2) in rh_mapping_pairs if j2 == j
        )
    # enforce fixed 

In [ ]:
# # --- Objective: Minimize total allocations ---
# model += lpSum(
#     variable for allocation_dict in [Allocation1, Allocation2, Allocation3, Allocation4]
#     for variable in allocation_dict.values()
# ), "Minimize_Total_Allocation"
model.writeLP("model.lp")
# --- Solve ---
status = model.solve(CPLEX_CMD(options=['set mip tolerances mipgap 0.01']))
print("Status:", LpStatus[model.status])

# # --- Output variables ---
# for allocation_dict in [Allocation1, Allocation2, Allocation3, Allocation4]:
#     for (i, j), variable in allocation_dict.items():
#         val = variable.varValue
#         if val is not None and val > 0:
#             print(f"{variable.name} = {val}")

PulpSolverError: PuLP: cannot execute cplex.exe

: 

In [10]:


# After solving the model
print("Objective Value =", value(model.objective))


Objective Value = 992337251.9706087


In [11]:
for v in model.variables():
    val = v.value()
    if val is not None and val > 0:
        print(v.name, val)

W_H01020642_Thiruvallur_531_Thiruvallur_Wheat 9180.0
W_H01020672_Thiruvallur_981_Thiruvallur_Wheat 13387.5
W_H010207714_Thiruvallur_484_Thiruvallur_Wheat 16406.830000000005
W_H010207714_Thiruvallur_981_Thiruvallur_Wheat 6543.169999999998
W_H01040422_Thiruvallur_531_Thiruvallur_Wheat 24862.500000000004
W_H01040582_Thiruvallur_531_Thiruvallur_Wheat 7650.0
W_H01040792_Thiruvallur_531_Thiruvallur_Wheat 3262.600000000002
W_H01040792_Thiruvallur_B28_Thiruvallur_Wheat 19687.4
W_H01040793_Thiruvallur_534_Thiruvallur_Wheat 19125.000000000004
W_H01040794_Thiruvallur_534_Thiruvallur_Wheat 18609.99
W_H01040794_Thiruvallur_B28_Thiruvallur_Wheat 2810.0099999999984
W_H01040952_Thiruvallur_531_Thiruvallur_Wheat 7650.000000000001
W_H01041053_Thiruvallur_531_Thiruvallur_Wheat 15299.999999999998
W_H01041142_Thiruvallur_17_Kancheepuram_Wheat 19125.0
W_H01041692_Thiruvallur_531_Thiruvallur_Wheat 20655.0
W_H01041772_Thiruvallur_534_Thiruvallur_Wheat 12240.0
W_H01041961_Thiruvallur_17_Kancheepuram_Wheat 2222

In [12]:
# Output_File = open('Backend//Inter_District1.csv', 'w')
# for v in model.variables():
#     if v.value() > 0:
#         Output_File.write(v.name + '\t' + str(v.value()) + '\n')

# Output_File = open('Backend//Inter_District1.csv', 'w')
# for v in model.variables():
#     if v.value() > 0:
#         Output_File.write(v.name + '\t' + str(v.value()) + '\n')

In [13]:
# Open two output files
gen_file = open('Backend//Generation_Variables.csv', 'w')
other_file = open('Backend//Inter_District1.csv', 'w')

# Loop through model variables
for v in model.variables():
    if v.value() is not None and v.value() > 0:
        if "Gen" in v.name:  # Adjust this condition based on your variable naming
            gen_file.write(v.name + '\t' + str(v.value()) + '\n')
        else:
            other_file.write(v.name + '\t' + str(v.value()) + '\n')
gen_file = open('Backend//Generation_Variables.csv', 'w')
other_file = open('Backend//Inter_District1.csv', 'w')

# Loop through model variables
for v in model.variables():
    if v.value() is not None and v.value() > 0:
        if "Gen" in v.name:  # Adjust this condition based on your variable naming
            gen_file.write(v.name + '\t' + str(v.value()) + '\n')
        else:
            other_file.write(v.name + '\t' + str(v.value()) + '\n')

In [14]:
df9 = pd.read_csv('Backend//Inter_District1.csv', header=None, encoding='latin1')

df9.columns = ['Tagging']
df9[[
    'Var',
    'WH_ID',
    'W_D',
    'FPS_ID',
    'FPS_D',
    'commodity_Value',
    ]] = df9[df9.columns[0]].str.split('_', n=6, expand=True)
del df9[df9.columns[0]]
df9[['commodity', 'Values']] = df9['commodity_Value'
        ].str.split('\\t', n=1, expand=True)
del df9['commodity_Value']
df9 = df9.drop(np.where(df9['commodity'] == 'Wheat1')[0])

def convert_to_numeric(value):
    try:
        return pd.to_numeric(value)
    except ValueError:
        return value


df9['WH_ID'] = df9['WH_ID'].apply(convert_to_numeric)
df9['FPS_ID'] = df9['FPS_ID'].apply(convert_to_numeric)

df9.to_excel('Backend//Tagging_Sheet_Pre.xlsx', sheet_name='BG_FPS')
# df31 = pd.read_excel('Backend//Tagging_Sheet_Pre.xlsx')

In [15]:
import pandas as pd

# --- Read the optimized tagging output ---
df = pd.read_excel("Backend/Tagging_Sheet_Pre.xlsx")

# --- Read all location sheets from Data_11.xlsx ---
data_path = "Backend/Data_11.xlsx"

# >>>> EDIT THIS DICTIONARY ONLY <<<<
# Keys are just logical names. Values are actual sheet names in your Excel file.
sheets = {
    "3WH": "A.3 Storage",
    "2WH": "A.3 RH",
    "1WH": "A.1 Warehouse",
    "SW": "A.2 Warehouse",
    "FPS": "A.2 FPS"
}

# >>>> ALSO edit ID column names if different in your sheets <<<<
id_map = {
    "3WH": "WH_ID",
    "2WH": "WH_ID",
    "1WH": "WH_ID",
    "SW": "SW_ID",
    "FPS": "FPS_ID"
}

node_frames = {}
for key, sheet in sheets.items():
    temp = pd.read_excel(data_path, sheet_name=sheet)
    temp.columns = [c.strip() for c in temp.columns]

    # normalize ID formats to string
    for c in temp.columns:
        if "ID" in c:
            temp[c] = temp[c].astype(str).str.strip()
    node_frames[key] = temp

# ------------------------------------------------------------
# 4. Combine all node sheets into one mapping table
# ------------------------------------------------------------
mapping_list = []
for key, df_map in node_frames.items():
    id_col = id_map[key]
    name_col = [c for c in df_map.columns if c.endswith("_Name")][0]
    lat_col  = [c for c in df_map.columns if c.endswith("_Lat")][0]
    lon_col  = [c for c in df_map.columns if c.endswith("_Long")][0]
    print(f"{key}: id_col={id_col}, name_col={name_col}, lat_col={lat_col}, lon_col={lon_col}")

    sub = df_map[[id_col, name_col, lat_col, lon_col]].copy()
    sub.columns = ["ID", "Name", "Lat", "Long"]
    mapping_list.append(sub)

mapping = pd.concat(mapping_list, ignore_index=True).drop_duplicates(subset="ID")

for c in df.columns:
    if "ID" in c:
        df[c] = df[c].astype(str).str.strip()

# ------------------------------------------------------------
# 6. Merge FROM and TO information
# ------------------------------------------------------------
# From side (WH_ID in solver output)
df = df.merge(mapping, left_on="WH_ID", right_on="ID", how="left")
df.rename(columns={"Name": "From_Name", "Lat": "From_Lat", "Long": "From_Long"}, inplace=True)
df.drop(columns="ID", inplace=True)

# To side (FPS_ID in solver output)
df = df.merge(mapping, left_on="FPS_ID", right_on="ID", how="left")
df.rename(columns={"Name": "To_Name", "Lat": "To_Lat", "Long": "To_Long"}, inplace=True)
df.drop(columns="ID", inplace=True)

# 7. Add metadata and rename for final output
# ------------------------------------------------------------
route_map = {
    "X": ("PC", "Mill"),
    "Y": ("PC", "Storage"),
    "Z": ("Storage", "Mill"),
    "W": ("Mill", "Warehouse"),
    "A": ("PC", "Railhead"),
    "B": ("Railhead", "Storage"),
    "C": ("Railhead", "Mill")
}
df["From"], df["To"] = zip(*df["Var"].map(route_map))
df["Scenario"] = "Optimized"
df["From_State"] = "Tamil Nadu"
df["To_State"] = "Tamil Nadu"

df.rename(columns={
    "W_D": "From_District",
    "WH_ID": "From_ID",
    "FPS_D": "To_District",
    "FPS_ID": "To_ID",
    "commodity": "commodity",
    "Values": "quantity"
}, inplace=True)

# ------------------------------------------------------------
# 8. Reorder and export
# ------------------------------------------------------------
final_cols = [
    "Scenario",
    "From", "From_State", "From_District", "From_ID", "From_Name",
    "From_Lat", "From_Long",
    "To", "To_State", "To_District", "To_ID", "To_Name",
    "To_Lat", "To_Long",
    "commodity", "quantity"
]
df_final = df[[c for c in final_cols if c in df.columns]]
df_final.to_excel("Backend/Tagging_Sheet_Converted.xlsx", index=False)

print("✅ Tagging sheet enriched and saved: Backend/Tagging_Sheet_Converted.xlsx")


3WH: id_col=WH_ID, name_col=WH_Name, lat_col=WH_Lat, lon_col=WH_Long
2WH: id_col=WH_ID, name_col=WH_Name, lat_col=WH_Lat, lon_col=WH_Long
1WH: id_col=WH_ID, name_col=WH_Name, lat_col=WH_Lat, lon_col=WH_Long
SW: id_col=SW_ID, name_col=SW_Name, lat_col=SW_Lat, lon_col=SW_Long
FPS: id_col=FPS_ID, name_col=FPS_Name, lat_col=FPS_Lat, lon_col=FPS_Long
✅ Tagging sheet enriched and saved: Backend/Tagging_Sheet_Converted.xlsx


In [16]:
import pandas as pd

# --- Read the optimized tagging output ---
df = pd.read_excel("Backend/Tagging_Sheet_Pre.xlsx")

# --- Read all location sheets from Data_11.xlsx ---
data_path = "Backend/Data_11.xlsx"

# >>>> EDIT THIS DICTIONARY ONLY <<<<
# Keys are just logical names. Values are actual sheet names in your Excel file.
sheets = {
    "3WH": "A.3 Storage",
    "2WH": "A.3 RH",
    "1WH": "A.1 Warehouse",
    "SW": "A.2 Warehouse",
    "FPS": "A.2 FPS"
}

# >>>> ALSO edit ID column names if different in your sheets <<<<
id_map = {
    "3WH": "WH_ID",
    "2WH": "WH_ID",
    "1WH": "WH_ID",
    "SW": "SW_ID",
    "FPS": "FPS_ID"
}

node_frames = {}
for key, sheet in sheets.items():
    temp = pd.read_excel(data_path, sheet_name=sheet)
    temp.columns = [c.strip() for c in temp.columns]

    # normalize ID formats to string
    for c in temp.columns:
        if "ID" in c:
            temp[c] = temp[c].astype(str).str.strip()
    node_frames[key] = temp

# ------------------------------------------------------------
# 4. Combine all node sheets into one mapping table
# ------------------------------------------------------------
mapping_list = []
for key, df_map in node_frames.items():
    id_col = id_map[key]
    name_col = [c for c in df_map.columns if c.endswith("_Name")][0]
    lat_col  = [c for c in df_map.columns if c.endswith("_Lat")][0]
    lon_col  = [c for c in df_map.columns if c.endswith("_Long")][0]
    print(f"{key}: id_col={id_col}, name_col={name_col}, lat_col={lat_col}, lon_col={lon_col}")

    sub = df_map[[id_col, name_col, lat_col, lon_col]].copy()
    sub.columns = ["ID", "Name", "Lat", "Long"]
    mapping_list.append(sub)

mapping = pd.concat(mapping_list, ignore_index=True).drop_duplicates(subset="ID")

for c in df.columns:
    if "ID" in c:
        df[c] = df[c].astype(str).str.strip()

# ------------------------------------------------------------
# 6. Merge FROM and TO information
# ------------------------------------------------------------
# From side (WH_ID in solver output)
df = df.merge(mapping, left_on="WH_ID", right_on="ID", how="left")
df.rename(columns={"Name": "From_Name", "Lat": "From_Lat", "Long": "From_Long"}, inplace=True)
df.drop(columns="ID", inplace=True)

# To side (FPS_ID in solver output)
df = df.merge(mapping, left_on="FPS_ID", right_on="ID", how="left")
df.rename(columns={"Name": "To_Name", "Lat": "To_Lat", "Long": "To_Long"}, inplace=True)
df.drop(columns="ID", inplace=True)

# 7. Add metadata and rename for final output
# ------------------------------------------------------------
route_map = {
    "X": ("PC", "Mill"),
    "Y": ("PC", "Storage"),
    "Z": ("Storage", "Mill"),
    "W": ("Mill", "Warehouse"),
    "A": ("PC", "Railhead"),
    "B": ("Railhead", "Storage"),
    "C": ("Railhead", "Mill")
}
df["From"], df["To"] = zip(*df["Var"].map(route_map))
df["Scenario"] = "Optimized"
df["From_State"] = "Tamil Nadu"
df["To_State"] = "Tamil Nadu"

df.rename(columns={
    "W_D": "From_District",
    "WH_ID": "From_ID",
    "FPS_D": "To_District",
    "FPS_ID": "To_ID",
    "commodity": "commodity",
    "Values": "quantity"
}, inplace=True)

# ------------------------------------------------------------
# 8. Reorder and export
# ------------------------------------------------------------
final_cols = [
    "Scenario",
    "From", "From_State", "From_District", "From_ID", "From_Name",
    "From_Lat", "From_Long",
    "To", "To_State", "To_District", "To_ID", "To_Name",
    "To_Lat", "To_Long",
    "commodity", "quantity"
]
df_final = df[[c for c in final_cols if c in df.columns]]
df_final.to_excel("Backend/Tagging_Sheet_Converted.xlsx", index=False)

print("✅ Tagging sheet enriched and saved: Backend/Tagging_Sheet_Converted.xlsx")


3WH: id_col=WH_ID, name_col=WH_Name, lat_col=WH_Lat, lon_col=WH_Long
2WH: id_col=WH_ID, name_col=WH_Name, lat_col=WH_Lat, lon_col=WH_Long
1WH: id_col=WH_ID, name_col=WH_Name, lat_col=WH_Lat, lon_col=WH_Long
SW: id_col=SW_ID, name_col=SW_Name, lat_col=SW_Lat, lon_col=SW_Long
FPS: id_col=FPS_ID, name_col=FPS_Name, lat_col=FPS_Lat, lon_col=FPS_Long
✅ Tagging sheet enriched and saved: Backend/Tagging_Sheet_Converted.xlsx


In [17]:
import math
import numpy as np
import json
import pickle
import os.path
from os import path
import shutil
import subprocess
import pymongo
import uuid
import pandas as pd
from pulp import *
import excelrd
from flask import Flask, request, jsonify
from flask_cors import CORS
import mysql.connector
import requests
import time
import secrets
import string
from datetime import datetime
from pulp import LpStatus, LpStatusInfeasible, LpStatusUnbounded, LpStatusNotSolved, LpStatusUndefined
df = pd.read_excel('Backend/Tagging_Sheet_Converted.xlsx', sheet_name='Sheet1')

# --- Load Weighbridge master ---
wb = pd.read_excel('Backend/Data_11.xlsx', sheet_name='Weighbridge')

# --- Haversine function ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*R*math.atan2(math.sqrt(a), math.sqrt(1 - a))
# Convert to numeric (if they are strings)
wb["WB_Lat"] = pd.to_numeric(wb["WB_Lat"], errors="coerce")
wb["WB_Long"] = pd.to_numeric(wb["WB_Long"], errors="coerce")

# Also ensure your from/to coordinates are numeric
df["From_Lat"] = pd.to_numeric(df["From_Lat"], errors="coerce")
df["From_Long"] = pd.to_numeric(df["From_Long"], errors="coerce")
df["To_Lat"] = pd.to_numeric(df["To_Lat"], errors="coerce")
df["To_Long"] = pd.to_numeric(df["To_Long"], errors="coerce")

# --- Build results list ---
results = []

for i, row in df.iterrows():
    from_lat, from_lon = row["From_Lat"], row["From_Long"]
    to_lat, to_lon = row["To_Lat"], row["To_Long"]

    # WB nearest to From
    wb_near_from = wb.loc[((wb["WB_Lat"] - from_lat)**2 + (wb["WB_Long"] - from_lon)**2).idxmin()]
    dist_from_wb = haversine(from_lat, from_lon, wb_near_from["WB_Lat"], wb_near_from["WB_Long"])
    dist_wb_to = haversine(wb_near_from["WB_Lat"], wb_near_from["WB_Long"], to_lat, to_lon)
    path1 = dist_from_wb + dist_wb_to

    # WB nearest to To
    wb_near_to = wb.loc[((wb["WB_Lat"] - to_lat)**2 + (wb["WB_Long"] - to_lon)**2).idxmin()]
    dist_from_wb2 = haversine(from_lat, from_lon, wb_near_to["WB_Lat"], wb_near_to["WB_Long"])
    dist_wb_to2 = haversine(wb_near_to["WB_Lat"], wb_near_to["WB_Long"], to_lat, to_lon)
    path2 = dist_from_wb2 + dist_wb_to2

    # Choose min path
    if path1 <= path2:
        chosen_wb = wb_near_from
        total_dist = path1
    else:
        chosen_wb = wb_near_to
        total_dist = path2

    # --- Final row format ---
    results.append({
        'Scenario': 'Optimized',
        'From': row['From'],
        'From_State': row.get('From_State', ''),      # if column missing, default blank
        'From_District': row.get('From_District', ''),
        'From_ID': row['From_ID'],
        'From_Name': row['From_Name'],
        'From_Lat': row['From_Lat'],
        'From_Long': row['From_Long'],

        'Via': 'WB',
        'Via_ID': chosen_wb['WB_ID'],
        'Via_Name': chosen_wb['WB_Name'],
        'Via_State': chosen_wb.get('WB_State', ''),
        'Via_District': chosen_wb['WB_District'],
        'Via_Lat': chosen_wb['WB_Lat'],
        'Via_Long': chosen_wb['WB_Long'],

        'To': row['To'],
        'To_ID': row['To_ID'],
        'To_Name': row['To_Name'],
        'To_State': row.get('To_State', ''),
        'To_District': row['To_District'],
        'To_Lat': row['To_Lat'],
        'To_Long': row['To_Long'],

        'Commodity': row.get('Commodity', 'Paddy'),
        'quantity': row['quantity'],

        'Distance': total_dist,
        'QKM': row['quantity'] * total_dist
    })

# --- Convert to DataFrame ---
df_final = pd.DataFrame(results)

# --- Save final tagging sheet with WB ---
df_final.to_excel('Backend/Tagging_Sheet_Pre3.xlsx', sheet_name='Sheet1', index=False)

print("✅ Final tagging sheet updated with weighbridge in the format you wanted.")

import pandas as pd
import requests
import time
import json

# --- Load weighbridge-tagged sheet ---
Result_Sheet1 = pd.ExcelFile("Backend//Tagging_Sheet_Pre3.xlsx")
df7 = pd.read_excel(Result_Sheet1, sheet_name="Sheet1")

# --- API credentials ---
auth_url = 'https://nctdelhi.pmgatishakti.gov.in/PMGatishaktiApiService/authenticate'
distance_url = 'https://nctdelhi.pmgatishakti.gov.in/PMGatishaktiApiService/dfpdapi/roaddistance'
auth_payload = {
    "username": "DFPD_C",
    "password": "W9Vtb8WKkt3"
}
FILE_PATH = 'distanceIndent.json'
BingMapsKey = "AirsAsdRojcMdt_sHviub_uETiraB-on6DR3Q_AVyrOCedF0FkITbpVVSDKx6hjT"

# --- Get Token ---
def get_token():
    response = requests.post(auth_url, json=auth_payload)
    if response.status_code == 200:
        token = response.json().get('token')
        return token if token else False
    return False

# --- Process batch through Gati Shakti ---
def process_batch(df_batch, from_cols, to_cols):
    token = get_token()
    time.sleep(10)  # API rate-limit gap
    headers = {'Authorization': f'Bearer {token}'}

    data = {
        "parameter": [{
            "src_lng": row[from_cols[1]],
            "src_lat": row[from_cols[0]],
            "dest_lng": row[to_cols[1]],
            "dest_lat": row[to_cols[0]]
        } for _, row in df_batch.iterrows()]
    }

    with open(FILE_PATH, 'w') as f:
        json.dump(data, f, indent=4)

    with open(FILE_PATH, 'rb') as f:
        files = {'LatsLongsFile': f}
        response = requests.post(distance_url, headers=headers, files=files)
        return response

# --- Fallback Bing Distance ---
def get_bing_distance(origin, dest):
    retries, max_retries = 0, 3
    while retries < max_retries:
        try:
            response = requests.get(
                f"https://dev.virtualearth.net/REST/v1/Routes/DistanceMatrix?"
                f"origins={origin}&destinations={dest}&travelMode=driving&key={BingMapsKey}"
            )
            resp = response.json()
            return resp['resourceSets'][0]['resources'][0]['results'][0]['travelDistance']
        except (requests.ConnectionError, requests.Timeout, KeyError, IndexError):
            retries += 1
            time.sleep(1)
    return -1

# --- Main distance calculation ---
batch_size = 30
total_rows = len(df7)
num_batches = (total_rows + batch_size - 1) // batch_size

dist_from_via, dist_via_to = [], []

for batch_num in range(num_batches):
    start_idx = batch_num * batch_size
    end_idx = min((batch_num + 1) * batch_size, total_rows)
    df_batch = df7.iloc[start_idx:end_idx]

    # 1️⃣ From → Via
    response = process_batch(df_batch,
                             from_cols=["From_Lat", "From_Long"],
                             to_cols=["Via_Lat", "Via_Long"])
    if response.status_code == 200:
        response_json = response.json()
        if 'data' in response_json and all('distance' in r for r in response_json['data']):
            dist_from_via.extend([r['distance'] for r in response_json['data']])
        else:
            for _, row in df_batch.iterrows():
                dist = get_bing_distance(f"{row['From_Lat']},{row['From_Long']}",
                                         f"{row['Via_Lat']},{row['Via_Long']}")
                dist_from_via.append(dist)
    else:
        for _, row in df_batch.iterrows():
            dist = get_bing_distance(f"{row['From_Lat']},{row['From_Long']}",
                                     f"{row['Via_Lat']},{row['Via_Long']}")
            dist_from_via.append(dist)

    # 2️⃣ Via → To
    response = process_batch(df_batch,
                             from_cols=["Via_Lat", "Via_Long"],
                             to_cols=["To_Lat", "To_Long"])
    if response.status_code == 200:
        response_json = response.json()
        if 'data' in response_json and all('distance' in r for r in response_json['data']):
            dist_via_to.extend([r['distance'] for r in response_json['data']])
        else:
            for _, row in df_batch.iterrows():
                dist = get_bing_distance(f"{row['Via_Lat']},{row['Via_Long']}",
                                         f"{row['To_Lat']},{row['To_Long']}")
                dist_via_to.append(dist)
    else:
        for _, row in df_batch.iterrows():
            dist = get_bing_distance(f"{row['Via_Lat']},{row['Via_Long']}",
                                     f"{row['To_Lat']},{row['To_Long']}")
            dist_via_to.append(dist)

# --- Final dataframe update ---
df7["Distance_From_Via"] = dist_from_via
df7["Distance_Via_To"] = dist_via_to
df7["QKM"] = df7["quantity"] * (df7["Distance_From_Via"] + df7["Distance_Via_To"])

# --- Save final result ---
df7 = df7.loc[:, [
    'Scenario',
    'From', 'From_State', 'From_District', 'From_ID', 'From_Name', 'From_Lat', 'From_Long',
    'Via', 'Via_ID', 'Via_Name', 'Via_State', 'Via_District', 'Via_Lat', 'Via_Long',
    'To', 'To_ID', 'To_Name', 'To_State', 'To_District', 'To_Lat', 'To_Long',
    'Commodity', 'quantity', 'Distance_From_Via', 'Distance_Via_To', 'QKM'
]]

df7.to_excel('Backend//Result_Sheet_With_wb.xlsx',
             sheet_name='Warehouse_FPS', index=False)

print("✅ Gati Shakti integration done: Distances split into From→Via and Via→To, with Bing fallback.")


C:\Users\Arvind Patidar\AppData\Local\Temp\ipykernel_24896\3697947926.py:60: FutureWarning: The behavior of Series.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  wb_near_to = wb.loc[((wb["WB_Lat"] - to_lat)**2 + (wb["WB_Long"] - to_lon)**2).idxmin()]


KeyError: nan

In [ ]:
# --- Split the result sheet (From→Via and Via→To) in same format ---

# 1️⃣ From → Via leg
df_from_via = pd.DataFrame({
    'Scenario': df7['Scenario'],
    'From': df7['From'],
    'From_State': df7['From_State'],
    'From_District': df7['From_District'],
    'From_ID': df7['From_ID'],
    'From_Name': df7['From_Name'],
    'From_Lat': df7['From_Lat'],
    'From_Long': df7['From_Long'],
    'To': df7['Via'],
    'To_ID': df7['Via_ID'],
    'To_Name': df7['Via_Name'],
    'To_State': df7['Via_State'],
    'To_District': df7['Via_District'],
    'To_Lat': df7['Via_Lat'],
    'To_Long': df7['Via_Long'],
    'commodity': df7['Commodity'],
    'quantity': df7['quantity'],
    'Distance': df7['Distance_From_Via']
})

# 2️⃣ Via → To leg
df_via_to = pd.DataFrame({
    'Scenario': df7['Scenario'],
    'From': df7['Via'],
    'From_State': df7['Via_State'],
    'From_District': df7['Via_District'],
    'From_ID': df7['Via_ID'],
    'From_Name': df7['Via_Name'],
    'From_Lat': df7['Via_Lat'],
    'From_Long': df7['Via_Long'],
    'To': df7['To'],
    'To_ID': df7['To_ID'],
    'To_Name': df7['To_Name'],
    'To_State': df7['To_State'],
    'To_District': df7['To_District'],
    'To_Lat': df7['To_Lat'],
    'To_Long': df7['To_Long'],
    'commodity': df7['Commodity'],
    'quantity': df7['quantity'],
    'Distance': df7['Distance_Via_To']
})

# 3️⃣ Combine both legs
df_split = pd.concat([df_from_via, df_via_to], ignore_index=True)

# --- Save the final split sheet ---
df_split.to_excel('Backend//Result_Sheet_Split_WB.xlsx', sheet_name='From_To_Via', index=False)

print("✅ Split sheet generated successfully: From→Via and Via→To, with same format.")


In [ ]:
# import pandas as pd

# # --- Read master Excel file ---
# USN = pd.ExcelFile('Backend//Data_11.xlsx')

# # --- Load all location master sheets ---
# FCI = pd.read_excel(USN, sheet_name='A.1 Warehouse')   # Procurement Centres (PC)
# FPS = pd.read_excel(USN, sheet_name='A.2 FPS')         # Fair Price Shops (Mills)
# SP  = pd.read_excel(USN, sheet_name='A.3 Storage')     # Storage points
# WH  = pd.read_excel(USN, sheet_name='A.2 Warehouse')   # Warehouses
# RH  = pd.read_excel(USN, sheet_name='A.3 RH')          # Railheads

# # --- Read tagging sheet to be enriched ---
# df = pd.read_excel('Backend//Tagging_Sheet_Pre.xlsx')

# # --- Map Var to route names ---
# route_map = {
#     "X": ("PC", "Mill"),
#     "Y": ("PC", "Storage"),
#     "Z": ("Storage", "Mill"),
#     "W": ("Mill", "Warehouse"),
#     "A": ("PC", "Railhead"),
#     "B": ("Railhead", "Storage"),
#     "C": ("Railhead", "Mill")
# }
# df["From"], df["To"] = zip(*df["Var"].map(route_map))

# # --- Add constants ---
# df["Scenario"] = "Optimized"
# df["From_State"] = "Tamil Nadu"
# df["To_State"] = "Tamil Nadu"

# # --- Rename columns to consistent form ---
# df.rename(columns={
#     "W_D": "From_District",
#     "WH_ID": "From_ID",
#     "FPS_D": "To_District",
#     "FPS_ID": "To_ID",
#     "commodity": "commodity",
#     "Values": "quantity"
# }, inplace=True)

# # --- Clean and standardize master sheets ---
# rename_map = {
#     "WH_ID": "ID", "WH_Name": "Name",
#     "FPS_ID": "ID", "FPS_Name": "Name",
#     "SW_ID": "ID", "SW_Name": "Name"
    
# }
# for df_master in [FCI, FPS, SP, WH, RH]:
#     df_master.rename(columns={k: v for k, v in rename_map.items() if k in df_master.columns}, inplace=True)

# # --- Define lat/long columns (try to auto-detect) ---
# for df_master in [FCI, FPS, SP, WH, RH]:
#     for col in df_master.columns:
#         if 'lat' in col.lower() and 'Lat' not in df_master.columns:
#             df_master.rename(columns={col: 'Lat'}, inplace=True)
#         if 'long' in col.lower() and 'Long' not in df_master.columns:
#             df_master.rename(columns={col: 'Long'}, inplace=True)

# # --- Master dictionary for dynamic lookup ---
# master_dict = {
#     "PC": FCI,
#     "Mill": FPS,
#     "Storage": SP,
#     "Warehouse": WH,
#     "Railhead": RH
# }

# # --- Function to add name and coordinates dynamically ---
# def add_location_info(df, prefix, entity_col, id_col):
#     df_out = df.copy()
#     for entity_name, master in master_dict.items():
#         mask = df_out[entity_col] == entity_name
#         if mask.any():
#             merge_cols = [c for c in ["ID", "Name", "Lat", "Long"] if c in master.columns]
#             merged = df_out.loc[mask].merge(
#                 master[merge_cols].rename(columns={
#                     "ID": id_col,
#                     "Name": f"{prefix}_Name",
#                     "Lat": f"{prefix}_Lat",
#                     "Long": f"{prefix}_Long"
#                 }),
#                 on=id_col,
#                 how="left"
#             )
#             # Assign merged values back
#             for c in [f"{prefix}_Name", f"{prefix}_Lat", f"{prefix}_Long"]:
#                 if c in merged.columns:
#                     df_out.loc[mask, c] = merged[c].values
#     return df_out

# # --- Apply to FROM and TO sides ---
# df = add_location_info(df, "From", "From", "From_ID")
# df = add_location_info(df, "To", "To", "To_ID")

# # --- Flag missing IDs for data cleaning ---
# df["From_Match_Missing"] = df["From_Name"].isna()
# df["To_Match_Missing"] = df["To_Name"].isna()

# # --- Reorder columns ---
# final_cols = [
#     "Scenario",
#     "From", "From_State", "From_District", "From_ID", "From_Name",
#     "From_Lat", "From_Long",
#     "To", "To_State", "To_District", "To_ID", "To_Name",
#     "To_Lat", "To_Long",
#     "commodity", "quantity",
#     "From_Match_Missing", "To_Match_Missing"
# ]
# df_final = df[[c for c in final_cols if c in df.columns]]

# # --- Save enriched tagging sheet ---
# output_path = "Tagging_Sheet_Converted_With_LatLong.xlsx"
# df_final.to_excel(output_path, index=False)

# print(f"✅ Final tagging sheet saved as {output_path}")
# print("📍 Included Lat/Long and Name columns for both From and To sides.")


In [ ]:
# from pulp import LpProblem, LpVariable, LpMinimize, lpSum, CPLEX_CMD, LpStatus
# import pandas as pd
# # Create the model
# model = LpProblem("Wheat_Distribution_Model", LpMinimize)

# objective = lpSum(Allocation1[(i, j)] * dist[(i, j)] for (i, j) in Allocation1) + \
#             lpSum(Allocation2[(i, j)] * dist1[(i, j)] for (i, j) in Allocation2) + \
#             lpSum(Allocation3[(i, j)] * dist3[(i, j)] for (i, j) in Allocation3) + \
#             lpSum(Allocation4[(i, j)] * dist2[(i, j)] for (i, j) in Allocation4)

# model += objective, "Total_Transport_Distance"
# a = 0.225  # coefficient

# # --- Rename columns as needed ---
# FCI.rename(columns={'Paddy_Procurement': 'Storage_Capacity'}, inplace=True)
# FPS.rename(columns={'Milling_Process': 'Allocation_Wheat', 'Mill_Capacity': 'Capacity'}, inplace=True)

# FPS['Demand'] = FPS['Allocation_Wheat']
# # Surplus and deficit districts
# # surplus_districts = set(['Thanjavur', 'Thiruvarur'])
# # deficit_districts = set(['Thiruchirappalli', 'Virudhunagar'])

# # All allocations
# # Define your surplus districts
# # List of all allocation variable sets
# # alloc_sets = [Allocation1, Allocation2, Allocation3, Allocation4]

# # for alloc in alloc_sets:
# #     keys_to_remove = []
# #     for key, var in alloc.items():
# #         var_name = str(var)
# #         parts = var_name.split("_")
# #         wh_district = parts[2]
# #         fps_district = parts[4]

# #         if not (wh_district in surplus_districts and fps_district in surplus_districts):
# #             model += var == 0
# #             keys_to_remove.append(key)
    
# #     # Actually remove them
# #     for k in keys_to_remove:
# #         del alloc[k]
# # print(Allocation1)
# # --- Constraint: Surplus flow rules ---
#     # FPS demand constraints
# # surplus_districts = set(['Thanjavur', 'Thiruvarur'])
# deficit_districts = ["ChennaiN", "ChennaiS", "Coimbatore", "Dharmapuri",
#                       "Dindigul", "Erode", "Kallakurichi", "Kanyakumari", "Karur", "Krishnagiri",
#                       "Madurai", "Namakkal", "Perambalur", "Salem", "Sivagangai", "Tenkasi", "Theni",
#                       "Thirunelveli", "Tiruppur", "Tirupathur", "Vellore", "Villupuram", "Virudhunagar",
#                       "Ramanathapuram", "Thoothukudi", "Nilgiris"]

# a = 0.225  # given coefficient

# # FPS demand constraints (only for FPS in surplus districts)
# for i in range(len(FPS['FPS_ID'])):
#     if FPS.loc[i, 'FPS_District'] in deficit_districts:
#         model += lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) <= a * FPS['Demand'][i]
#     else:
#         # Optionally block allocations to FPS in deficit districts
#         model += lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) == 0

# for i in range(len(FPS['FPS_ID'])):
#     fps_cap = FPS.loc[i, 'Capacity']  # milling capacity of FPS
#     # Total wheat allocated to FPS (from Allocation1 and Allocation3)
#     total_allocated_to_FPS = lpSum(Allocation1[key] for key in Allocation1 if key[1] == i) + \
#                              lpSum(Allocation3[key] for key in Allocation3 if key[1] == i)
    
#     # Constrain by milling capacity
#     model += total_allocated_to_FPS <= fps_cap


# # SP capacity constraints (only for SP in surplus districts)
# for i in range(len(SP['WH_ID'])):
#     if SP.loc[i, 'WH_District'] in deficit_districts:
#         model += lpSum(Allocation2[key] for key in Allocation2 if key[1] == i) <= SP['Capacity'][i]
#     else:
#         model += lpSum(Allocation2[key] for key in Allocation2 if key[1] == i) == 0

# # FCI storage capacity constraints (only for FCI in surplus districts)
# for i in range(len(FCI['WH_ID'])):
#     if FCI.loc[i, 'WH_District'] in deficit_districts:
#         model += (
#             lpSum(Allocation1[key] for key in Allocation1 if key[0] == i) +
#             lpSum(Allocation2[key] for key in Allocation2 if key[0] == i)
#         ) == FCI['Storage_Capacity'][i]
#     else:
#         model += (
#             lpSum(Allocation1[key] for key in Allocation1 if key[0] == i) +
#             lpSum(Allocation2[key] for key in Allocation2 if key[0] == i)
#         ) == 0

# # FPS allocation and warehouse demand relations (only FPS in surplus districts)
# for j in range(len(FPS['FPS_ID'])):
#     if FPS.loc[j, 'FPS_District'] in deficit_districts:
#         model += (
#             lpSum(Allocation4[key] for key in Allocation4 if key[0] == j)
#         ) == 0.67 * (
#             lpSum(Allocation1[key] for key in Allocation1 if key[1] == j) +
#             lpSum(Allocation3[key] for key in Allocation3 if key[1] == j)
#         )
#     else:
#         model += lpSum(Allocation4[key] for key in Allocation4 if key[0] == j) == 0

# # SP to FPS flow limit (only for SP in surplus districts)
# for k in range(len(SP['WH_ID'])):
#     if SP.loc[k, 'WH_District'] in deficit_districts:
#         model += (
#             lpSum(Allocation3[key] for key in Allocation3 if key[0] == k)
#         ) == (
#             lpSum(Allocation2[key] for key in Allocation2 if key[1] == k) +
#             Generation[k]
            
            
#         )
#     else:
#         model += lpSum(Allocation3[key] for key in Allocation3 if key[0] == k) == 0
#         model += Generation[k] == 0

# # Warehouse demand fulfillment (only for warehouses in surplus districts)
# for m in range(len(WH['SW_ID'])):
#     if WH.loc[m, 'SW_District'] in deficit_districts:
#         model += (
#             lpSum(Allocation4[key] for key in Allocation4 if key[1] == m)
#         ) == WH['Annual_Demand'][m]
#     else:
#         model += lpSum(Allocation4[key] for key in Allocation4 if key[1] == m) == 0


In [ ]:
# status = model.solve(CPLEX_CMD(options=['set mip tolerances mipgap 0.01']))
# print("Status:", LpStatus[model.status])

In [ ]:
# # Open two output files
# gen_file = open('Backend//Generation_Variables.csv', 'w')
# other_file = open('Backend//Inter_District2.csv', 'w')

# # Loop through model variables
# for v in model.variables():
#     if v.value() > 0:
#         if "Gen" in v.name:  # Adjust this condition based on your variable naming
#             gen_file.write(v.name + '\t' + str(v.value()) + '\n')
#         else:
#             other_file.write(v.name + '\t' + str(v.value()) + '\n')
# gen_file = open('Backend//Generation_Variables.csv', 'w')
# other_file = open('Backend//Inter_District2.csv', 'w')
# gen_file = open('Backend//Generation_Variables.csv', 'w')
# other_file = open('Backend//Inter_District2.csv', 'w')

# # Loop through model variables
# for v in model.variables():
#     if v.value() > 0:
#         if "Gen" in v.name:  # Adjust this condition based on your variable naming
#             gen_file.write(v.name + '\t' + str(v.value()) + '\n')
#         else:
#             other_file.write(v.name + '\t' + str(v.value()) + '\n')
# gen_file = open('Backend//Generation_Variables.csv', 'w')
# other_file = open('Backend//Inter_District2.csv', 'w')

# # # Loop through model variables
# # for v in model.variables():
# #     if v.value() > 0:
# #         if "Gen" in v.name:  # Adjust this condition based on your variable naming
# #             gen_file.write(v.name + '\t' + str(v.value()) + '\n')
# #         else:
# #             other_file.write(v.name + '\t' + str(v.value()) + '\n')



In [ ]:
# df9 = pd.read_csv('Backend//Inter_District2.csv', header=None, encoding='latin1')

# df9.columns = ['Tagging']
# df9[[
#     'Var',
#     'WH_ID',
#     'W_D',
#     'FPS_ID',
#     'FPS_D',
#     'commodity_Value',
#     ]] = df9[df9.columns[0]].str.split('_', n=6, expand=True)
# del df9[df9.columns[0]]
# df9[['commodity', 'Values']] = df9['commodity_Value'
#         ].str.split('\\t', n=1, expand=True)
# del df9['commodity_Value']
# df9 = df9.drop(np.where(df9['commodity'] == 'Wheat1')[0])

# def convert_to_numeric(value):
#     try:
#         return pd.to_numeric(value)
#     except ValueError:
#         return value


# df9['WH_ID'] = df9['WH_ID'].apply(convert_to_numeric)
# df9['FPS_ID'] = df9['FPS_ID'].apply(convert_to_numeric)

# df9.to_excel('Backend//Tagging_Sheet_Pre2.xlsx', sheet_name='BG_FPS')
# # df31 = pd.read_excel('Backend//Tagging_Sheet_Pre.xlsx')

In [ ]:
# import pandas as pd

# # Read your tagging sheet
# df = pd.read_excel('Backend//Tagging_Sheet_Pre2.xlsx')

# # Map Var values to route names
# route_map = {
#     "X": ("PC", "Mill"),
#     "Y": ("PC", "Storage"),
#     "Z": ("Storage", "Mill"),
#     "W": ("Mill", "Warehouse")
# }
# df.loc[df["Var"].isin(route_map), ["From", "To"]] = df.loc[df["Var"].isin(route_map), "Var"].map(route_map).tolist()

# # # Apply route mapping
# # df["From"], df["To"] = zip(*df["Var"].map(route_map))

# # Add constant columns
# df["Scenario"] = "Optimized"
# df["From_State"] = "Tamil Nadu"
# df["To_State"] = "Tamil Nadu"

# # Rename for standard structure
# df.rename(columns={
#     "W_D": "From_District",
#     "WH_ID": "From_ID",
#     "FPS_D": "To_District",
#     "FPS_ID": "To_ID",
#     "commodity": "commodity",
#     "Values": "quantity"
# }, inplace=True)

# # Add empty placeholders for name & coordinates
# df["From_Name"] = ""
# df["To_Name"] = ""
# df["From_Lat"] = ""
# df["From_Long"] = ""
# df["To_Lat"] = ""
# df["To_Long"] = ""

# # Reorder columns
# df_final = df[[
#     "Scenario", "From", "From_State", "From_District", "From_ID", "From_Name",
#     "From_Lat", "From_Long", "To", "To_ID", "To_Name", "To_State",
#     "To_District", "To_Lat", "To_Long", "commodity", "quantity"
# ]]

# # Save the converted data
# df_final.to_excel("Tagging_Sheet_Converted2.xlsx", index=False)
# print("✅ Tagging sheet successfully converted and saved as Tagging_Sheet_Converted2.xlsx")


In [ ]:
# df9 = pd.read_csv('Backend//Inter_District2.csv', header=None, encoding='latin1')

# df9.columns = ['Tagging']
# df9[[
#     'Var',
#     'WH_ID',
#     'W_D',
#     'FPS_ID',
#     'FPS_D',
#     'commodity_Value',
#     ]] = df9[df9.columns[0]].str.split('_', n=6, expand=True)
# del df9[df9.columns[0]]
# df9[['commodity', 'Values']] = df9['commodity_Value'
#         ].str.split('\\t', n=1, expand=True)
# del df9['commodity_Value']
# df9 = df9.drop(np.where(df9['commodity'] == 'Wheat1')[0])

# def convert_to_numeric(value):
#     try:
#         return pd.to_numeric(value)
#     except ValueError:
#         return value

# # 
# df9['WH_ID'] = df9['WH_ID'].apply(convert_to_numeric)
# df9['FPS_ID'] = df9['FPS_ID'].apply(convert_to_numeric)

# df9.to_excel('Backend//Tagging_Sheet_Pre2.xlsx', sheet_name='BG_FPS')
# df31 = pd.read_excel('Backend//Tagging_Sheet_Pre2.xlsx')